# 9주차 · 기본 정렬 구현 및 비교
## 학생 배포용: TODO 구현·테스트·성능 실험
데이터구조 | 임화정 | Python 3.8+

학습목표: 버블·선택·삽입 정렬을 구현하고, 불변식·안정성·시간/보조공간 복잡도를 설명하며 동일 입력의 성능을 측정한다.

강의계획서의 9주차 목표와 2주차 Big-O, 7주차 BST/TreeSort를 연결한다. 기존 자료의 개념→구현→검증→성능 분석 흐름을 따른다.

사용법: 위에서 아래로 실행한다. 학생용은 미구현 함수를 안내하고 해당 실험을 건너뛴다. TODO 완료 뒤 구현·테스트·실험 셀을 다시 실행한다.

권장 150분 수업: 개념/추적 50분, 구현 45분, 실험 35분, 토론/정리 20분(휴식 별도).

In [ ]:
import random
import time
import statistics
import csv
from collections import Counter
SEED = 2026
# Python 3.8+ / Colab / Jupyter. 그래프 셀만 matplotlib가 필요합니다.
print("9주차 실습 준비 완료")

## 1. 공통 인터페이스와 사전 예측
`sort(a, key=lambda x:x)`는 리스트 자체를 오름차순으로 변경하고 동일한 리스트를 반환한다. 반환값 복사 금지. `sorted`/`list.sort`는 구현에 사용하지 않고 정답 검증에만 사용한다.

버블은 인접 교환, 선택은 최소값 선택, 삽입은 정렬된 접두부에 삽입한다. 핵심 구현의 보조공간은 O(1); 실험용 복사·추적 기록은 별도 O(n) 이상이다. key 호출/비교는 O(1)이라고 가정한다.

**예측**: 이미 정렬된 입력에 가장 유리한 것은? 선택 정렬도 빨라질까?

## bubble_sort 구현
각 패스 뒤 최댓값이 오른쪽에 확정됩니다. end를 줄이고 교환이 없으면 종료. > 비교는 동점 순서를 유지합니다.

In [ ]:
def bubble_sort(a, key=lambda x: x):
    # TODO: 위 설명과 슬라이드의 불변식을 참고하여 구현
    raise NotImplementedError("bubble_sort 구현 필요")

## selection_sort 구현
i 이전 접두부가 확정됩니다. min_idx를 i로 초기화하고 나머지 구간을 모두 비교합니다. 최솟값과 한 번 교환. 일반 교환형 선택 정렬은 불안정합니다.

In [ ]:
def selection_sort(a, key=lambda x: x):
    # TODO: 위 설명과 슬라이드의 불변식을 참고하여 구현
    raise NotImplementedError("selection_sort 구현 필요")

## insertion_sort 구현
a[:i]가 정렬되어 있습니다. current를 저장하고 큰 원소만 오른쪽으로 이동합니다. j >= 0을 먼저 검사하여 음수 인덱스 접근을 막습니다.

In [ ]:
def insertion_sort(a, key=lambda x: x):
    # TODO: 위 설명과 슬라이드의 불변식을 참고하여 구현
    raise NotImplementedError("insertion_sort 구현 필요")

## 2. 정확성 검사
빈 입력·단일 원소·중복·음수·정렬/역순·고정 시드 무작위 입력. 결과 순서와 원소 보존, in-place 반환 계약을 확인한다.

In [ ]:
algorithms = {"Bubble": bubble_sort, "Selection": selection_sort, "Insertion": insertion_sort}
def test_sort(fn):
    cases = [[], [1], [2, 1], [1, 2, 3], [3, 2, 1], [2, 2, 1, 2], [-3, 0, -1, 5]]
    rng = random.Random(SEED)
    cases += [[rng.randrange(-20, 21) for _ in range(n)] for n in range(41)]
    for original in cases:
        a = original.copy()
        returned = fn(a)
        assert returned is a, "반환값은 입력 리스트 자체여야 합니다."
        assert a == sorted(original), (original, a)
        assert Counter(a) == Counter(original)
    return len(cases)
ready = {}
for name, fn in algorithms.items():
    try:
        print(name, test_sort(fn), "개 테스트 통과")
        ready[name] = fn
    except NotImplementedError:
        print(name, "TODO 구현 후 이 셀을 다시 실행하세요.")
assert len(ready) in range(4)

## 3. 패스 추적
[5, 1, 4, 2, 8]의 각 패스 결과를 먼저 손으로 작성하라.
학생 답안: 버블 ___ / 선택 ___ / 삽입 ___

## 4. 안정성 실험
동일 키의 원래 상대 순서가 유지되는가? 아래 기록을 첫 번째 필드만으로 정렬하고 A/B 순서를 관찰하라. 선택 정렬이 불안정한 이유를 설명하라.

In [ ]:
records = [(2, "A"), (2, "B"), (1, "C")]
for name, fn in ready.items():
    a = records.copy()
    fn(a, key=lambda r: r[0])
    print(name, a, "동점 순서 유지:", [r[1] for r in a if r[0] == 2] == ["A", "B"])
# 튜플 전체가 아니라 첫 번째 필드만 비교해야 안정성을 검증할 수 있습니다.

## 5. 연산 횟수와 복잡도
아래 계측기는 키 비교만 comparisons로 센다. 인덱스 검사 제외. swap은 두 원소 교환 1회, shift는 삽입 정렬의 오른쪽 이동 1회다. swap과 shift는 서로 같은 비용 단위가 아니며 전체 쓰기 횟수도 아니다. 계측기 함수는 측정용 모형이며 실행 시간 비교에는 사용하지 않는다.

In [ ]:
def count_operations(data, kind):
    a = data.copy()
    comparisons = swaps = shifts = 0
    n = len(a)
    if kind == "Bubble":
        for end in range(n - 1, 0, -1):
            changed = False
            for j in range(end):
                comparisons += 1
                if a[j] > a[j + 1]:
                    a[j], a[j + 1] = a[j + 1], a[j]
                    swaps += 1
                    changed = True
            if not changed: break
    elif kind == "Selection":
        for i in range(n - 1):
            m = i
            for j in range(i + 1, n):
                comparisons += 1
                if a[j] < a[m]: m = j
            if m != i:
                a[i], a[m] = a[m], a[i]
                swaps += 1
    elif kind == "Insertion":
        for i in range(1, n):
            current = a[i]
            j = i - 1
            while j >= 0:
                comparisons += 1
                if a[j] <= current: break
                a[j + 1] = a[j]
                shifts += 1
                j -= 1
            a[j + 1] = current
    else:
        raise ValueError(kind)
    assert a == sorted(data)
    return {"comparisons": comparisons, "swaps": swaps, "shifts": shifts}
for kind in algorithms:
    for label, data in [("sorted", list(range(8))), ("reverse", list(range(7, -1, -1)))]:
        print(kind, label, count_operations(data, kind))
assert count_operations(list(range(8)), "Bubble")["comparisons"] == 7
assert count_operations(list(range(8)), "Selection")["comparisons"] == 28
assert count_operations(list(range(7, -1, -1)), "Insertion")["shifts"] == 28

### 분석 질문
1. 선택 정렬 비교 횟수 n(n−1)/2를 유도하라.
2. 조기 종료 버블의 최선 Θ(n), 최악 Θ(n²)를 설명하라.
3. 삽입 정렬의 shift 횟수와 역전쌍 수의 관계는?
4. n=8 정렬/역순에서 각 비교·교환·이동 횟수를 기록하라.

학생 답안: ___

## 6. 공정한 실행 시간 측정
5개 입력 유형 × 4개 크기 × 3개 알고리즘. 고정 시드·동일 데이터·실행 전 복사·워밍업·5회 반복·중앙값. 시계는 perf_counter_ns. 생성/복사/검증/I/O 비용 제외. 키 함수는 공통으로 사용하므로 키 호출 오버헤드가 포함된다. 시간이 작은 경우 잡음이 크다. 환경마다 수치는 달라지며 한 번의 실측으로 Big-O를 증명할 수 없다. 거의 정렬된 입력은 인접 교환으로 생성된다.

In [ ]:
def make_data(n, pattern, seed):
    rng = random.Random(seed)
    if pattern == "random": return [rng.randrange(n * 10 + 1) for _ in range(n)]
    if pattern == "sorted": return list(range(n))
    if pattern == "reverse": return list(range(n - 1, -1, -1))
    if pattern == "duplicates": return [rng.randrange(5) for _ in range(n)]
    if pattern == "nearly_sorted":
        a = list(range(n))
        for _ in range(max(1, n // 20)):
            if n > 1:
                j = rng.randrange(n - 1)
                a[j], a[j + 1] = a[j + 1], a[j]
        return a
    raise ValueError(pattern)
SIZES = [100, 200, 400, 800]  # 처음에는 작게; 큰 n은 O(n²)로 오래 걸립니다.
PATTERNS = ["random", "sorted", "reverse", "nearly_sorted", "duplicates"]
REPEATS = 5
results = []
if len(ready) != 3:
    print("세 알고리즘 구현 및 테스트를 완료하면 성능 비교가 실행됩니다.")
else:
    for pattern in PATTERNS:
        for n in SIZES:
            data = make_data(n, pattern, SEED + n)
            expected = sorted(data)  # 정답 생성은 측정 밖
            rng = random.Random(SEED + n)
            samples = {name: [] for name in ready}
            for fn in ready.values(): fn(data.copy())  # 워밍업
            for _ in range(REPEATS):
                names = list(ready)
                rng.shuffle(names)  # 실행 순서 편향 완화
                for name in names:
                    a = data.copy()  # 복사 비용은 측정 제외
                    start = time.perf_counter_ns()
                    ready[name](a)
                    elapsed = time.perf_counter_ns() - start
                    assert a == expected  # 검증도 측정 제외
                    samples[name].append(elapsed / 1_000_000)
            for name, values in samples.items():
                results.append({"pattern": pattern, "n": n, "algorithm": name,
                                "median_ms": statistics.median(values),
                                "min_ms": min(values), "max_ms": max(values)})
    with open("benchmark.csv", "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0]))
        writer.writeheader(); writer.writerows(results)
    for row in results[:9]: print(row)
    print("전체 결과: week09_benchmark.csv", len(results), "행")

## 7. 성능 시각화
아래 그래프는 현재 실행에서 측정한 중앙값만 사용한다. 범례와 단위를 확인하고 유형별로 비교하라.

In [ ]:
if results:
    try:
        import matplotlib.pyplot as plt
        try:
            import koreanize_matplotlib
        except ImportError:
            pass  # 그래프 라벨은 영문이므로 Colab에서도 실행 가능
        colors = ["#1751D0", "#E2661E", "#0E9E8F"]
        fig, axes = plt.subplots(2, 3, figsize=(15, 8), facecolor="#FDFDFF")
        for ax, pattern in zip(axes.flat, PATTERNS):
            ax.set_facecolor("#FDFDFF")
            for color, name in zip(colors, algorithms):
                rows = sorted([r for r in results if r["pattern"] == pattern and r["algorithm"] == name], key=lambda r:r["n"])
                ax.plot([r["n"] for r in rows], [r["median_ms"] for r in rows], marker="o", color=color, label=name)
            ax.set(title=pattern, xlabel="Input size n", ylabel="Median time (ms)")
            ax.grid(axis="y", color="#E2E6F0"); ax.set_axisbelow(True)
            ax.legend(frameon=False)
        axes.flat[-1].axis("off")
        fig.tight_layout()
        fig.savefig("benchmark.png", dpi=150)
        plt.show()
    except ImportError:
        print("matplotlib가 없어 그래프를 건너뜁니다. Colab에서 실행하거나 CSV를 활용하세요.")
else:
    print("구현 완료 후 성능 측정 셀부터 다시 실행하세요.")

## 8. 결과 분석 및 제출
- 무작위/역순에서 n을 두 배로 늘릴 때 시간 비율 T(2n)/T(n)은?
- 정렬 입력의 버블·삽입과 선택은 어떻게 다른가?
- 안정성과 이동 횟수는 어떤 문제에서 중요한가?
- nearly_sorted 결과를 역전쌍 관점에서 설명하라.
- 이론과 실측이 다른 이유 2가지를 적어라.

**답안**:

실측 표 요약 ___ / 그래프 해석 ___ / 복잡도 근거 ___ / 선택 근거 ___

제출: DS_09배포용.ipynb, benchmark.csv, benchmark.png, 분석문 포함.
lms 과제란에 제출

## 이론 요약 및 근거 자료
|알고리즘|최선|평균|최악|보조공간|안정성|
|---|---|---|---|---|---|
|버블(조기 종료)|Θ(n)|Θ(n²)|Θ(n²)|O(1)|안정|
|선택(교환형)|Θ(n²)|Θ(n²)|Θ(n²)|O(1)|불안정|
|삽입(이동형)|Θ(n)|Θ(n²)|Θ(n²)|O(1)|안정|
평균은 무작위 순열 등 일반적인 입력 가정. 본 주차 코드는 수업 목표에 맞춰 새로 작성하였다.

근거: 데이터구조 강의계획서(26-2) 2~3쪽, 02. 알고리즘 복잡도와 Big-O(성능 측정·시간/공간 구분), 07 이진 탐색 트리(BST)(불변식·트레이스·TreeSort·실측), 01/03/04/05/06 강의자료의 이론→Python 구현→검증 흐름.